# 🔍🏭 GarudaChip — Lint & LibreLane Hardening (standalone notebook)

This notebook reproduces the **two physical gates** GarudaChip's pipeline runs *after*
a design simulates — **Verilator structural lint** and **LibreLane RTL → GDSII
hardening** — on a single, simple example design, the way the LibreLane Colab notebook
does it, but using **GarudaChip's own flow and tools** (Verilator + LibreLane CLI +
the local `gf180mcuD` PDK).

It is also the **regression test** for the lint path bug: the Verilator include dir
must be passed **attached** (`-I<dir>`, no space) — passing it as a separate argument
made Verilator treat the `rtl/` path as a top module and fail with
*"Cannot find file containing module: …/rtl"*.

**Design under test:** `examples/verilog_designs/generated_pwm_generator/` — a 61-line
parameterised PWM generator (`pwm_generator.v`) plus its self-checking testbench
(`pwm_generator_tb.v`). It is the simplest design in the example set: one clean module,
no headers, no multi-dimensional ports — so it lints clean and hardens to GDSII.

**Flow:** locate tools → show RTL → **Verilator lint** → iverilog sanity sim →
write LibreLane `config.json` → **run LibreLane** → collect GDSII + metrics.


## 0 · Locate the tools and stage the example design

In [2]:
import os, glob, shutil, subprocess, json
from pathlib import Path

# --- Find the repo root (walk up until we see examples/verilog_designs) ----------
here = Path.cwd().resolve()
REPO = next((p for p in (here, *here.parents)
             if (p / "examples" / "verilog_designs").is_dir()), here)
print("Repo root :", REPO)

# --- Locate verilator (same fallback as app.py: PATH -> env -> Nix store) --------
def find_verilator():
    v = shutil.which("verilator")
    if v:
        return v
    v = os.getenv("VERILATOR_BIN")
    if v and Path(v).exists():
        return v
    cands = sorted(glob.glob("/nix/store/*verilator*/bin/verilator"))
    return cands[-1] if cands else None

VERILATOR = find_verilator()
LIBRELANE = os.getenv("LIBRELANE_BIN", "librelane")
PDK       = os.getenv("PDK", "gf180mcuD")
PDK_ROOT  = os.getenv("PDK_ROOT", os.path.expanduser("~/.ciel"))

print("verilator :", VERILATOR or "NOT FOUND")
print("librelane :", shutil.which(LIBRELANE) or "NOT FOUND")
print("PDK       :", PDK, "@", PDK_ROOT,
      "(exists)" if Path(PDK_ROOT, PDK).exists() else "(MISSING)")

# --- Stage the example design into a clean working dir under output/ -------------
SRC_DESIGN = REPO / "examples" / "verilog_designs" / "generated_pwm_generator"
TOP        = "pwm_generator"

WORK = REPO / "output" / "nb_lint_harden"
RTL, TB, CHIP = WORK / "rtl", WORK / "tb", WORK / "chip"
if WORK.exists():
    shutil.rmtree(WORK)
for d in (RTL, TB):
    d.mkdir(parents=True, exist_ok=True)

shutil.copy(SRC_DESIGN / f"{TOP}.v",     RTL / f"{TOP}.v")
shutil.copy(SRC_DESIGN / f"{TOP}_tb.v",  TB  / f"{TOP}_tb.v")
print("\nStaged design ->", WORK)
print(" ", *(str(p.relative_to(WORK)) for p in sorted(WORK.rglob("*.v"))))

Repo root : /home/irman/GarudaChip
verilator : /nix/store/wl8c1s56dn566asz404bqwcl0gdrw933-verilator-5.044/bin/verilator
librelane : /home/irman/.nix-profile/bin/librelane
PDK       : gf180mcuD @ /home/irman/.ciel (exists)

Staged design -> /home/irman/GarudaChip/output/nb_lint_harden
  rtl/pwm_generator.v tb/pwm_generator_tb.v


## 1 · The design we're hardening

The RTL (`pwm_generator.v`) — a counter compared against `compare`/`period` config inputs.

In [3]:
print((RTL / f"{TOP}.v").read_text())

/*
 * Module: pwm_generator
 *
 * Description:
 * This module generates a Pulse Width Modulated (PWM) signal. The frequency and
 * duty cycle of the PWM signal can be configured dynamically. The core of the
 * module is a counter that increments on each clock cycle. The PWM output is
 * determined by comparing the counter's value against a 'compare' value (for
 * duty cycle) and a 'period' value (for frequency).
 *
 * The PWM frequency is determined by: F_pwm = F_clk / (period + 1)
 * The Duty Cycle is determined by: Duty = compare / (period + 1)
 *
 * For example, with a 100MHz clock:
 * - To get a ~1kHz PWM signal, 'period' should be (100e6 / 1e3) - 1 = 99999.
 * - For a 25% duty cycle, 'compare' should be 99999 * 0.25 = 25000.
 *
 * The rising edge of the PWM output is aligned with the counter reset.
 */
module pwm_generator #(
    parameter COUNTER_WIDTH = 16
) (
    // System Inputs
    input wire clk,
    input wire rst,

    // Configuration Inputs
    input wire [COUNTER_WIDTH-

## 2 · Verilator structural lint  🔍

This is GarudaChip's lint gate: it fails the build on the exact structural issues that
later break synthesis — combinational loops (`UNOPTFLAT`), multiple-driver nets
(`MULTIDRIVEN`) and inferred latches (`LATCH`).

> **The path fix:** the include dir is passed as **`-I{rtl}` (attached, no space)**.
> Passing `"-I", str(RTL)` as two args makes Verilator read the directory path as a
> positional top-level source and abort with *"Cannot find file containing module:
> …/rtl"*. That was the lint failure that wasn't really a lint failure.


In [4]:
assert VERILATOR, "verilator not found — install it or set VERILATOR_BIN"
vfiles = sorted(glob.glob(str(RTL / "*.v")))

cmd = [VERILATOR, "--lint-only", "-Wno-fatal",
       "--Werror-MULTIDRIVEN", "--Werror-LATCH", "--Werror-UNOPTFLAT",
       f"-I{RTL}",                      # <-- ATTACHED include dir (the fix)
       "--top-module", TOP, *vfiles]

print("$", "verilator --lint-only --Werror-MULTIDRIVEN --Werror-LATCH "
      "--Werror-UNOPTFLAT -I<rtl> --top-module", TOP,
      *(os.path.basename(f) for f in vfiles), "\n")

proc = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
out = (proc.stdout + "\n" + proc.stderr).strip()
print(out or "(no output)")

lint_failed = proc.returncode != 0 or "%Error" in out
print("\n", "❌ LINT FAILED — would route to the corrector" if lint_failed
      else "✅ Lint clean — RTL is structurally sound for hardening.")

$ verilator --lint-only --Werror-MULTIDRIVEN --Werror-LATCH --Werror-UNOPTFLAT -I<rtl> --top-module pwm_generator pwm_generator.v 

- V e r i l a t i o n   R e p o r t: Verilator 5.044 2026-01-01 rev v5.040
- Verilator: Built from 0.038 MB sources in 2 modules, into 0.009 MB in 3 C++ files needing 0.000 MB
- Verilator: Walltime 0.019 s (elab=0.003, cvt=0.008, bld=0.000); cpu 0.009 s on 1 threads; alloced 19.824 MB

 ✅ Lint clean — RTL is structurally sound for hardening.


## 3 · Functional sanity sim (iverilog)  🧪 *(optional)*

Not part of hardening, but a quick confirmation the staged design actually works:
compile RTL + testbench with `iverilog` and look for `Result: PASSED`.


In [5]:
SIM = WORK / "sim"; SIM.mkdir(exist_ok=True)
sim_files = sorted(glob.glob(str(RTL / "*.v"))) + sorted(glob.glob(str(TB / "*.v")))

if not shutil.which("iverilog"):
    print("iverilog not on PATH — skipping the sanity sim.")
else:
    subprocess.run(["iverilog", "-g2005-sv", "-o", str(SIM / "design.vvp"),
                    "-I", str(RTL), *sim_files], check=True, timeout=90)
    run = subprocess.run(["vvp", "design.vvp"], cwd=SIM,
                         capture_output=True, text=True, timeout=90)
    tail = "\n".join(run.stdout.strip().splitlines()[-12:])
    print(tail)
    print("\n", "✅ Simulation passed." if "Result: PASSED" in run.stdout
          or ("ERROR" not in run.stdout and "Result: FAILED" not in run.stdout)
          else "❌ Simulation failed.")

Time=13965000 | rst=0 | period= 99 | compare=150 | counter= 91 | pwm_out=1
Time=13975000 | rst=0 | period= 99 | compare=150 | counter= 92 | pwm_out=1
Time=13985000 | rst=0 | period= 99 | compare=150 | counter= 93 | pwm_out=1
Time=13995000 | rst=0 | period= 99 | compare=150 | counter= 94 | pwm_out=1
Time=14005000 | rst=0 | period= 99 | compare=150 | counter= 95 | pwm_out=1
Time=14015000 | rst=0 | period= 99 | compare=150 | counter= 96 | pwm_out=1
Time=14025000 | rst=0 | period= 99 | compare=150 | counter= 97 | pwm_out=1
Time=14035000 | rst=0 | period= 99 | compare=150 | counter= 98 | pwm_out=1
Time=14045000 | rst=0 | period= 99 | compare=150 | counter= 99 | pwm_out=1

--- All test cases completed. Finishing simulation. ---
Time=14055000 | rst=0 | period= 99 | compare=150 | counter=  0 | pwm_out=1

 ✅ Simulation passed.


## 4 · LibreLane configuration  ⚙️

The same `config.json` GarudaChip writes for hardening, targeting the local
`gf180mcuD` PDK. Note the **lint-is-non-fatal** block: LibreLane runs Verilator
internally and would otherwise *quit* on a lint error; we already gate on lint
ourselves in step 2, so here we let those findings stay informational and reach GDSII.
Set `LIBRELANE_STRICT_LINT=1` to put the hard lint gate back.


In [6]:
if CHIP.exists():
    shutil.rmtree(CHIP)
csrc = CHIP / "src"; csrc.mkdir(parents=True, exist_ok=True)

design_files = []
for p in sorted(glob.glob(str(RTL / "*.v"))) + sorted(glob.glob(str(RTL / "*.vh"))):
    name = os.path.basename(p)
    shutil.copy(p, csrc / name)
    if name.endswith(".v"):
        design_files.append(f"dir::src/{name}")

CLOCK_PORT, CLOCK_PERIOD, DIE_UM, CORE_UTIL = "clk", 24.0, 600.0, 25
strict = os.getenv("LIBRELANE_STRICT_LINT", "0") in ("1", "true", "yes")

config = {
    "DESIGN_NAME": TOP, "VERILOG_FILES": design_files,
    "CLOCK_PORT": CLOCK_PORT, "CLOCK_PERIOD": CLOCK_PERIOD, "PDK": PDK,
    "FP_SIZING": "absolute", "DIE_AREA": [0, 0, DIE_UM, DIE_UM],
    "FP_CORE_UTIL": CORE_UTIL, "PL_TARGET_DENSITY_PCT": max(20, CORE_UTIL + 5),
    "PRIMARY_GDSII_STREAMOUT_TOOL": "klayout",
    **({} if strict else {
        "LINTER_ERROR_ON_LATCH": False,
        "LINTER_ERROR_ON_MULTIDRIVEN": False,
        "ERROR_ON_LINTER_ERRORS": False,
        "ERROR_ON_LINTER_WARNINGS": False,
        "LINTER_DISABLE_WARNINGS": [
            "UNOPTFLAT", "WIDTH", "WIDTHEXPAND", "WIDTHTRUNC", "WIDTHCONCAT",
            "CASEINCOMPLETE", "CASEOVERLAP", "UNUSEDSIGNAL", "UNDRIVEN",
            "IMPLICIT", "BLKSEQ", "SYNCASYNCNET", "DECLFILENAME", "EOFNEWLINE",
        ],
    }),
}
(CHIP / "config.json").write_text(json.dumps(config, indent=2))
print(json.dumps(config, indent=2))

{
  "DESIGN_NAME": "pwm_generator",
  "VERILOG_FILES": [
    "dir::src/pwm_generator.v"
  ],
  "CLOCK_PORT": "clk",
  "CLOCK_PERIOD": 24.0,
  "PDK": "gf180mcuD",
  "FP_SIZING": "absolute",
  "DIE_AREA": [
    0,
    0,
    600.0,
    600.0
  ],
  "FP_CORE_UTIL": 25,
  "PL_TARGET_DENSITY_PCT": 30,
  "PRIMARY_GDSII_STREAMOUT_TOOL": "klayout",
  "LINTER_ERROR_ON_LATCH": false,
  "LINTER_ERROR_ON_MULTIDRIVEN": false,
  "ERROR_ON_LINTER_ERRORS": false,
  "ERROR_ON_LINTER_WARNINGS": false,
  "LINTER_DISABLE_WARNINGS": [
    "UNOPTFLAT",
    "WIDTH",
    "WIDTHEXPAND",
    "WIDTHTRUNC",
    "WIDTHCONCAT",
    "CASEINCOMPLETE",
    "CASEOVERLAP",
    "UNUSEDSIGNAL",
    "UNDRIVEN",
    "IMPLICIT",
    "BLKSEQ",
    "SYNCASYNCNET",
    "DECLFILENAME",
    "EOFNEWLINE"
  ]
}


## 5 · Run LibreLane (RTL → synthesis → PnR → signoff → GDSII)  🏭

Runs the `librelane --manual-pdk --pdk-root <PDK_ROOT> config.json` CLI exactly as the
app does, streaming the log. This is the slow step (a few minutes on first run).


In [7]:
assert shutil.which(LIBRELANE), "librelane not found — install it or set LIBRELANE_BIN"
cmd = [LIBRELANE, "--manual-pdk", "--pdk-root", PDK_ROOT, "config.json"]
print("$", *cmd, "  (cwd:", CHIP, ")\n")

env = {**os.environ, "PDK_ROOT": PDK_ROOT}
lines = []
proc = subprocess.Popen(cmd, cwd=str(CHIP), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
for raw in proc.stdout:
    line = raw.rstrip()
    lines.append(line)
    print(line)          # live log
proc.wait()
(CHIP / "librelane.log").write_text("\n".join(lines))
print("\nLibreLane exit code:", proc.returncode)

$ librelane --manual-pdk --pdk-root /home/irman/.ciel config.json   (cwd: /home/irman/GarudaChip/output/nb_lint_harden/chip )

[19:58:17] INFO     Starting a new run of the 'Classic' flow with    ]8;id=433575;file:///nix/store/svai9q3r6sqydq8vqs393lbl2qkps0ay-python3.13-librelane-3.0.3/lib/python3.13/site-packages/librelane/flows/flow.py\flow.py]8;;\:]8;id=190025;file:///nix/store/svai9q3r6sqydq8vqs393lbl2qkps0ay-python3.13-librelane-3.0.3/lib/python3.13/site-packages/librelane/flows/flow.py#654\654]8;;\
                    the tag 'RUN_2026-06-10_19-58-17'.                          

Classic ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0/100 0:00:00
[19:58:17] INFO     Starting…                                  ]8;id=863275;file:///nix/store/svai9q3r6sqydq8vqs393lbl2qkps0ay-python3.13-librelane-3.0.3/lib/python3.13/site-packages/librelane/flows/sequential.py\sequential.py]8;;\:]8;id=258526;file:///nix/store/svai9q3r6sqydq8vqs393lbl2qkps0ay-python3.13-librelane-3.0.3/lib/pyt

## 6 · Collect the GDSII + metrics  🎉

In [8]:
gds = (sorted(glob.glob(str(CHIP / "runs" / "**" / "final" / "**" / "*.gds"), recursive=True))
       or sorted(glob.glob(str(CHIP / "runs" / "**" / "*.gds"), recursive=True)))

if gds:
    final = WORK / f"{TOP}.gds"
    shutil.copy(gds[-1], final)
    print(f"🎉 GDSII generated -> {final.relative_to(REPO)} "
          f"({final.stat().st_size/1024:.1f} KiB)")
else:
    print("No GDS produced — inspect the log above (chip/librelane.log).")

mp = sorted(glob.glob(str(CHIP / "runs" / "**" / "metrics.json"), recursive=True))
if mp:
    m = json.load(open(mp[-1]))
    show = {k: m[k] for k in (
        "design__die__area", "design__instance__count__stdcell",
        "timing__setup__ws", "clock__skew__worst") if k in m}
    print("\nKey metrics:")
    for k, v in show.items():
        print(f"  {k:42s} {v}")

🎉 GDSII generated -> output/nb_lint_harden/pwm_generator.gds (1441.2 KiB)

Key metrics:
  design__die__area                          360000
  design__instance__count__stdcell           2630
  timing__setup__ws                          6.336758852557827
